# 1단계: 마라톤 온라인 크롤링

- 대상: 서울, 2026년 11월·12월 대회 중 오늘 이후 대회
- 출처: 마라톤 온라인 일정 (http://www.roadrun.co.kr/schedule/list.php)
- 결과: `data/marathons_raw.csv`

In [1]:
import os
import re        # 정규표현식: 글자 속에서 원하는 패턴 찾기
import time      # 요청 사이에 잠깐 쉬기
from datetime import date

import pandas as pd
import requests                # 웹페이지 요청
from bs4 import BeautifulSoup  # HTML 파싱

BASE = 'http://www.roadrun.co.kr/schedule/'  # 마라톤 온라인 일정 주소
HEADERS = {'User-Agent': 'Mozilla/5.0'}      # 브라우저인 척 해야 차단 안 됨

## 1. 목록 페이지에서 대회 번호 가져오기

사이트 검색창이 POST 방식이라, 지역·연도·월을 form 데이터로 보냅니다.

In [2]:
def get_marathon_ids(year, month, area='서울'):
    # 사이트가 옛날 한글 인코딩(euc-kr)이라 '서울'도 euc-kr로 바꿔서 보냄
    data = {'area_key': area.encode('euc-kr'), 'syear_key': year, 'smonth_key': month}
    res = requests.post(BASE + 'list.php', data=data, headers=HEADERS)
    res.encoding = 'euc-kr'

    # 상세 링크가 'view.php?no=41790' 형태 → 숫자(대회 번호)만 뽑기
    ids = re.findall(r'view\.php\?no=(\d+)', res.text)
    return list(dict.fromkeys(ids))  # 같은 링크가 여러 번 나와서 중복 제거

ids = get_marathon_ids(2026, 11) + get_marathon_ids(2026, 12)
print(len(ids), '개 대회')

44 개 대회


## 2. 상세 페이지에서 대회 정보 가져오기

상세 페이지는 「항목명 | 값」 2칸짜리 표라서, 칸이 2개인 줄만 골라 `{항목명: 값}` 으로 저장합니다.

In [3]:
def get_marathon_detail(no):
    res = requests.get(BASE + 'view.php', params={'no': no}, headers=HEADERS)
    res.encoding = 'euc-kr'
    soup = BeautifulSoup(res.text, 'html.parser')

    info = {'번호': no}
    for tr in soup.find_all('tr'):                  # 표의 각 줄
        tds = tr.find_all('td', recursive=False)    # 그 줄의 칸들
        if len(tds) == 2:                           # 「항목명 | 값」 줄만
            key = tds[0].get_text(strip=True)
            info[key] = tds[1].get_text(' ', strip=True)
    return info

get_marathon_detail(ids[0])  # 하나만 먼저 확인

{'번호': '41871',
 '대회명': 'JUSTICE RUN SEOCHO 2026',
 '대표자명': '법률신문사',
 'E-mail': 'lawtimesmarathon@gmail.com',
 '대회일시': '2026년11월1일 출발시간:10km_07:30, 5km_7:40',
 '전화번호': '010-9573-7669',
 '대회종목': '10km,5km',
 '대회지역': '서울',
 '대회장소': '반포한강공원 달빛광장',
 '주최단체': '법률신문',
 '접수기간': '2026년9월21일~2026년10월21일',
 '홈페이지': 'http:// justicerun.kr/',
 '기타소개': 'JUSTICE RUN SEOCHO 2026 정의를 잇고, 미래를 달리다. 법률신문과 함께하는 제1회 JUSTICE RUN SEOCHO 2026'}

In [4]:
rows = []
for no in ids:
    rows.append(get_marathon_detail(no))
    time.sleep(1)  # 서버에 부담 안 주게 1초씩 쉬기

df = pd.DataFrame(rows)  # 딕셔너리 리스트 → 표
df.head()

,번호,대회명,대표자명,E-mail,대회일시,전화번호,대회종목,대회지역,대회장소,주최단체,접수기간,홈페이지,기타소개,대회장
0,41871,JUSTICE RUN SEOCHO 2026,법률신문사,lawtimesmarathon@gmail.com,"2026년11월1일 출발시간:10km_07:30, 5km_7:40",010-9573-7669,"10km,5km",서울,반포한강공원 달빛광장,법률신문,2026년9월21일~2026년10월21일,http:// justicerun.kr/,"JUSTICE RUN SEOCHO 2026 정의를 잇고, 미래를 달리다. 법률신문과...",NaN
1,41790,26서울동행마라톤대회,유범진,kesyf@naver.com,2026년11월1일 출발시간:7:30,02-736-1136,"하프,10km,5km",서울,서울 목동종합운동장 주경기장,서울동행마라톤조직위원회,2026년8월10일~2026년10월2일,http:// seoulrun.com,깊어가는 가을 이봉주와 함께 안양천을 함께 뛰어요,
2,41259,2026 jtbc 마라톤,류영호,runable@joins.com,2026년11월1일 출발시간:08:00,02-2031-1935,"풀,10km",서울,상암동 평화의 광장,jtbc,2026년4월13일~2026년4월15일,http:// marathon.jtbc.com/,"참가비 풀코스 15만원, 10km 10만원",
3,41819,SaveRace 2026,김연수,emara@daum.net,2026년11월7일 출발시간:07:30,02-1566-1936,10km,서울,여의도공원 문화의마당,"카카오뱅크, 유니셰프",2026년9월1일~2026년10월25일,http:// www.saverace.co.kr,10km를 달리는 Save Race 2026에 동참해 지구와 미래 세대를 지켜주세요...,
4,41815,넥슨과 함께하는 2026 푸르메런,주상은,jak@e-world21.co.kr,2026년11월7일 출발시간:9시,02-338-0344,"10km,3km",서울,상암 월드컵공원 평화광장,푸르메재단,2026년8월26일~2026년9월23일,http:// purmerun.com/,"푸르메재단 넥슨어린이재활병원 10주년 기념, 넥슨과 함께하는 2026 미라클365 ...",


## 3. 오늘 이후 대회만 남기기

In [5]:
# '2026년11월1일 출발시간:7:30' → 앞의 '2026년11월1일'만 뽑아서 날짜로 변환
day_text = df['대회일시'].str.extract(r'(\d+년\s*\d+월\s*\d+일)')[0]
df['날짜'] = pd.to_datetime(day_text, format='%Y년%m월%d일')

df = df[df['대회지역'] == '서울']                     # 서울 대회만
df = df[df['날짜'] >= pd.Timestamp(date.today())]     # 오늘 이후만
df = df[~df['대회명'].str.contains('취소')]            # 취소된 대회 빼기
df = df.sort_values('날짜').reset_index(drop=True)

print(len(df), '개 대회 남음')
df[['날짜', '대회명', '대회종목', '대회장소']]

39 개 대회 남음


,날짜,대회명,대회종목,대회장소
0,2026-11-07,SaveRace 2026,10km,여의도공원 문화의마당
1,2026-11-07,넥슨과 함께하는 2026 푸르메런,"10km,3km",상암 월드컵공원 평화광장
2,2026-11-08,강남하프마라톤(제17회강남구청장배육상대회),"하프,10km",강남구 대치유수지 체육공원
3,2026-11-08,제3회 자유민주 마라톤대회,"10km,6.1km",서울시청
4,2026-11-08,2026 서울구석구석 국제 트레일런,트레일런 14km,서울마당(지하철1호선 시청역 4번출구)
5,2026-11-08,제11회 송파구청장배 마라톤,"10km,5km",송파구 여성축구장
6,2026-11-14,제2회 안양천 단풍길 마라톤,"하프,10km,5km",신정교 하부 영롱이 억새구장
7,2026-11-14,제2회 한강 브릿지 런,65km,구리한강시민공원 주차장
8,2026-11-14,서울 빵빵런 2026,"10km,5km",뚝섬한강공원 수변무대
9,2026-11-14,2026 통큰런,"10km,5km,3km(가족런)",상암 월드컵공원 평화광장


In [6]:
os.makedirs('data', exist_ok=True)  # data 폴더가 없으면 만들기
df.to_csv('data/marathons_raw.csv', index=False, encoding='utf-8-sig')  # utf-8-sig: 엑셀에서 한글 안 깨짐